In [1]:
import numpy as np
import cv2

In [ ]:


def create_lowpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            if distance > cutoff_freq:
                mask[i, j] = 0
    return mask

def create_highpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            if distance < cutoff_freq:
                mask[i, j] = 0
    return mask

def create_bandpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    inner_cutoff = cutoff_freq - 10
    outer_cutoff = cutoff_freq + 10
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            if distance < inner_cutoff or distance > outer_cutoff:
                mask[i, j] = 0
    return mask

def create_bandstop_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    inner_cutoff = cutoff_freq - 10
    outer_cutoff = cutoff_freq + 10
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            if inner_cutoff <= distance <= outer_cutoff:
                mask[i, j] = 0
    return mask

def create_laplacian_mask(rows, cols, center_row, center_col, cutoff_freq):
    inner_cutoff = cutoff_freq - 10
    outer_cutoff = cutoff_freq + 10
    crow, ccol = rows // 2, cols // 2
    u = np.arange(rows).reshape(-1, 1) - crow
    v = np.arange(cols).reshape(1, -1) - ccol
    D2 = u**2 + v**2
    mask = -4 * np.pi**2 * D2
    return mask

def create_gaussian_lowpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            mask[i, j] = np.exp(-(distance**2) / (2 * (cutoff_freq**2)))
    return mask

def create_gaussian_highpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            mask[i, j] = 1 - np.exp(-(distance**2) / (2 * (cutoff_freq**2)))
    return mask

def create_butterworth_lowpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    n = 2
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            if distance == 0:
                mask[i, j] = 1
            else:
                mask[i, j] = 1 / (1 + (distance / cutoff_freq)**(2*n))
    return mask

def create_butterworth_highpass_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    n = 2
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            if distance == 0:
                mask[i, j] = 0
            else:
                mask[i, j] = 1 / (1 + (cutoff_freq / distance)**(2*n))
    return mask

def create_notch_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    for i in range(rows):
        for j in range(cols):
            distance1 = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            distance2 = np.sqrt((i - (rows - center_row))**2 + (j - (cols - center_col))**2)
            if distance1 < cutoff_freq or distance2 < cutoff_freq:
                mask[i, j] = 0
    return mask

def create_homomorphic_mask(rows, cols, center_row, center_col, cutoff_freq):
    mask = np.ones((rows, cols), np.float32)
    gamma_l = 0.5
    gamma_h = 2.0
    c = 1
    for i in range(rows):
        for j in range(cols):
            distance = np.sqrt((i - center_row)**2 + (j - center_col)**2)
            mask[i, j] = (gamma_h - gamma_l) * (1 - np.exp(-c * (distance**2) / (cutoff_freq**2))) + gamma_l
    return mask

def create_filter_mask(rows, cols, center_row, center_col, filter_type, cutoff_freq, cutoff_freq2=None):
    filter_functions = {
        'lowpass': create_lowpass_mask,
        'highpass': create_highpass_mask,
        'bandpass': create_bandpass_mask,
        'bandstop': create_bandstop_mask,
        'laplacian': create_laplacian_mask,
        'gaussian_lowpass': create_gaussian_lowpass_mask,
        'gaussian_highpass': create_gaussian_highpass_mask,
        'butterworth_lowpass': create_butterworth_lowpass_mask,
        'butterworth_highpass': create_butterworth_highpass_mask,
        'notch': create_notch_mask,
        'homomorphic': create_homomorphic_mask
    }
    
    if filter_type in filter_functions:
        return filter_functions[filter_type](rows, cols, center_row, center_col, cutoff_freq)
    else:
        raise ValueError(f"Unknown filter type: {filter_type}")

In [ ]:

def apply_filter_to_image(img1, filter_type, cutoff_freq, cutoff_freq2=None):
    img_float32 = np.float32(img1)
    dft = cv2.dft(img_float32, flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)
    rows, cols = img1.shape
    center_row, center_col = rows // 2, cols // 2
    filter_mask = create_filter_mask(rows, cols, center_row, center_col, filter_type, cutoff_freq, cutoff_freq2)
    
    # Apply filter to both real and imaginary parts
    if filter_type == 'laplacian':
        # For Laplacian, it's applied differently as it's already in frequency domain
        filtered_shift = dft_shift * filter_mask[:, :, np.newaxis]
    else:
        # For other filters, apply the mask to both channels
        filtered_shift = dft_shift * filter_mask[:, :, np.newaxis]
    
    filtered_ishift = np.fft.ifftshift(filtered_shift)
    img_back = cv2.idft(filtered_ishift)
    img_filtered = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])
    img_filtered = cv2.normalize(img_filtered, None, 0, 255, cv2.NORM_MINMAX)
    img_filtered = np.uint8(img_filtered)
    
    return img_filtered, filter_mask

# Example usage:
if __name__ == "__main__":
    # Read image in grayscale
    img1 = cv2.imread('your_image.jpg', cv2.IMREAD_GRAYSCALE)
    
    if img1 is None:
        print("Error: Could not load image")
        exit()
    
    # Apply different filters
    cutoff = 30  # Adjust cutoff frequency as needed
    
    # Lowpass filter
    lowpass_result, lowpass_mask = apply_filter_to_image(img1, 'lowpass', cutoff)
    
    # Highpass filter
    highpass_result, highpass_mask = apply_filter_to_image(img1, 'highpass', cutoff)
    
    # Gaussian Lowpass filter
    gaussian_lowpass_result, gaussian_lowpass_mask = apply_filter_to_image(img1, 'gaussian_lowpass', cutoff)
    
    # Butterworth Highpass filter
    butterworth_highpass_result, butterworth_highpass_mask = apply_filter_to_image(img1, 'butterworth_highpass', cutoff)
    
    # Bandpass filter
    bandpass_result, bandpass_mask = apply_filter_to_image(img1, 'bandpass', cutoff)
    
    # Display results
    cv2.imshow('Original Image', img1)
    cv2.imshow('Lowpass Filtered', lowpass_result)
    cv2.imshow('Highpass Filtered', highpass_result)
    cv2.imshow('Gaussian Lowpass', gaussian_lowpass_result)
    cv2.imshow('Butterworth Highpass', butterworth_highpass_result)
    cv2.imshow('Bandpass', bandpass_result)
    
    # Display filter masks (normalized for visualization)
    cv2.imshow('Lowpass Mask', cv2.normalize(lowpass_mask, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
    cv2.imshow('Highpass Mask', cv2.normalize(highpass_mask, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Save results if needed
    cv2.imwrite('lowpass_result.jpg', lowpass_result)
    cv2.imwrite('highpass_result.jpg', highpass_result)
    cv2.imwrite('gaussian_lowpass_result.jpg', gaussian_lowpass_result)